In [2]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re

/tmp/ipykernel_5873/4287803411.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


## Create your chroma client & Select Model 
- we use persistence to store it in memory such that even when we restart the machine or device we still have the document in place

- **Context** is the text you paste into the prompt so the model can answer from your document.

In [25]:
chroma_client = chromadb.PersistentClient(path="CHROMA")
MODEL = "llama3.2:3b"
CONTEXT  =""
QUESTION = ""
PERSONA = f""" 
Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""


## Load your documents

In [4]:
loader = PyPDFLoader("data/10201979.pdf")
pages = loader.load()

# Cleaning text

In [5]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

In [6]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

# Chunking

higher chunk size less granularity 
higher chunk overlap more repetition hence  try and reduce the chunk over lap size to get better context


In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 100) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

# Embeddings

In [8]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [9]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [10]:
ids = [ f"10201979:{i}" for i in range(len(chunks))]
documents = [chunk.page_content for chunk in chunks ]
metadatas = [{"source": chunk.metadata.get("source", " "),  }for chunk in chunks ]
embeddings = doc_embeddings


confirming the sizes of each element to assertain consistency

In [11]:
print(len(ids), len(documents), len(metadatas), len(embeddings))  # all four equal
print(len(embeddings))                                         # single number, e.g. 768

65 65 65 65
65


In [12]:
collection_research = chroma_client.get_or_create_collection(name="CHROMA")

In [13]:
len(embeddings)

65

Find out the difference between *upsert* and *add* ad highlight the respective usecase

In [14]:
collection_research.add(
    ids = ids, 
    documents = documents, 
    metadatas = metadatas, 
    embeddings =  embeddings 
)

In [18]:
response = collection_research.query(
    query_texts=["What is a recipe?"],
    n_results=2,
    query_embeddings = [createEmbedding("What is a recipe?")],
    # include=["documents", "metadatas", "distances"],
)
 

In [21]:
response

{'ids': [['10201979:10', '10201979:39']],
 'embeddings': None,
 'documents': [['scratch?”), to utter novices: \nI’m a 19 year old guy living in Singapore with nothing to do for 2 weeks. I am thinkin g of \nbaking/cooking for fun! What should i know/have before i start on my adventure? What \nare the tools involved? How do I bake a cookie? \n \nExamining Recipes and Their Interpretation \n \n Another approach to supporting information behaviour in cooking begins with a close examination \nof information presentation structures in recipes, and how people understand (or misunderstand) them. \n Hamada et al (2000) and Shirai & Ookawa (2006) analyse collections of Japanese recipes to \ndevelop models of the steps involved in cooking, with the goal of automated processing existing recipes',
   'want to know how much powdered cocoa or unsweetened baker’s squares to substitute.”). This category \nalso includes queries relating to the calories or dietary units in food items and substitution of 

In [38]:
response["documents"]

[['scratch?”), to utter novices: \nI’m a 19 year old guy living in Singapore with nothing to do for 2 weeks. I am thinkin g of \nbaking/cooking for fun! What should i know/have before i start on my adventure? What \nare the tools involved? How do I bake a cookie? \n \nExamining Recipes and Their Interpretation \n \n Another approach to supporting information behaviour in cooking begins with a close examination \nof information presentation structures in recipes, and how people understand (or misunderstand) them. \n Hamada et al (2000) and Shirai & Ookawa (2006) analyse collections of Japanese recipes to \ndevelop models of the steps involved in cooking, with the goal of automated processing existing recipes',
  'want to know how much powdered cocoa or unsweetened baker’s squares to substitute.”). This category \nalso includes queries relating to the calories or dietary units in food items and substitution of food items \n(eg, Weight Watchers diet ‘points’) and substitution of food item

# Retreival 

Recall what we did on pure ollama chat? i.e ollama.chat(model.., messages..., options)? now basd on that we use and guide the model to use our vector DB as out source of truth

In [41]:
#  query vector DB 
def retreiveFromVectorDB(query, k):
    response = collection_research.query(
    query_texts=[query],
    n_results=k,
    query_embeddings = [createEmbedding(query)],
    # include=["documents", "metadatas", "distances"],
    )
    return response["documents"][0]

def answer(question, k=4):
    chunks = retreiveFromVectorDB(question, k=k)
    CONTEXT = "\n\n---\n\n".join(chunks)  
    QUESTION = question
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""

    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},   # keeps answers grounded and repeatable (lower chance of hallucination and creativity)
    )
    return resp.message.content


In [42]:
print(answer("What is a recipe?"))

A recipe is defined as a minimal form consisting of a title, a list of ingredients, the amount of each ingredient, and a set of instructions for processing the ingredients to prepare the named dish.


**Find out:** How do temperature and model choice affect a RAG system's answers?